# Architecture Diagram

## Mục tiêu

Phần sơ đồ kiến trúc tổng hợp thiết kế hoàn chỉnh của Lab 04 dựa trên đề bài `[BigData] Lab04 - StreamingV0.pdf` và hiện trạng mã nguồn trong repository. Trọng tâm là pipeline streaming tăng dần: Parser Service publish event qua Kafka, Neo4j Kafka Sink ghi graph topology trực tiếp bằng Cypher `MERGE`, và Spark Structured Streaming ghi metadata sang MongoDB bằng upsert có checkpoint.

## Yêu cầu từ đề bài

| Nhóm yêu cầu | Cách hệ thống đáp ứng |
|---|---|
| Repository cloning và file discovery | Shallow clone repository `huggingface/transformers-pr-agent`, sau đó lọc các file Python hợp lệ. |
| Incremental CPG Parser Service | Parser xử lý từng file độc lập, dùng `ast` chuẩn của Python để sinh AST, CFG, DFG và Call graph. |
| Kafka topic layout | Tách riêng `cpg.nodes`, `cpg.edges`, `source.metadata`, `parser.errors` và `connector.errors`. |
| Neo4j graph ingestion | Kafka Connect Sink đọc node/edge events và ghi graph trực tiếp vào Neo4j bằng Cypher `MERGE`, kèm uniqueness constraints theo stable ID. |
| MongoDB metadata ingestion | Spark Structured Streaming consume `source.metadata`, dùng checkpoint và upsert document theo `file_id`. |
| Idempotent replay | Stable deterministic IDs, SQLite state store, graph diff, Neo4j `MERGE`/delete idempotent và MongoDB upsert tạo replay không trùng lặp. |

## Sơ đồ kiến trúc tổng thể

```mermaid
graph TD
    SourceRepo["Source Repository<br/>huggingface/transformers-pr-agent"] -->|"shallow clone"| Discovery["Source Discovery CLI"]
    Discovery -->|"eligible .py files"| Parser["CPG Parser Service"]

    subgraph ParserCore["Parser Core"]
        Parser --> Ast["AST Builder"]
        Parser --> Cfg["CFG Builder"]
        Parser --> Dfg["DFG Builder"]
        Parser --> Call["Call Graph Builder"]
        Parser --> Metadata["Metadata Extractor"]
        Parser --> StableId["Stable ID Generator"]
        Parser --> State[("SQLite State Store")]
    end

    Parser -->|"NODE_UPSERT / NODE_DELETE"| NodesTopic["Kafka topic: cpg.nodes"]
    Parser -->|"EDGE_UPSERT / EDGE_DELETE"| EdgesTopic["Kafka topic: cpg.edges"]
    Parser -->|"FILE_METADATA_UPSERT"| MetadataTopic["Kafka topic: source.metadata"]
    Parser -->|"PARSER_ERROR"| ParserErrors["Kafka topic: parser.errors"]

    NodesTopic --> Neo4jSink["Neo4j Kafka Sink Connector"]
    EdgesTopic --> Neo4jSink
    Neo4jSink -->|"Cypher MERGE / MATCH DELETE"| Neo4j[("Neo4j Graph Database")]
    Neo4jSink -.->|"connector failure"| ConnectorErrors["Kafka topic: connector.errors"]

    MetadataTopic --> Spark["Spark Structured Streaming"]
    Spark -->|"checkpointed upsert"| Mongo[("MongoDB file_statistics")]
```

## Luồng replay tăng dần

```mermaid
sequenceDiagram
    autonumber
    actor User as Workflow / CLI
    participant Repo as Source Repository
    participant Parser as Parser Service
    participant SQLite as SQLite State Store
    participant Kafka as Kafka
    participant Neo4j as Neo4j Sink
    participant Spark as Spark Streaming
    participant Mongo as MongoDB

    User->>Repo: Modify one Python file
    User->>Parser: replay-file --file path.py
    Parser->>SQLite: Load previous content_hash and graph IDs
    Parser->>Repo: Read current file bytes
    Parser->>Parser: Parse AST, CFG, DFG, Call graph
    Parser->>Parser: Compute graph diff
    Parser->>Kafka: Publish DELETE events for stale IDs
    Parser->>Kafka: Publish UPSERT events for current graph
    Parser->>Kafka: Publish FILE_METADATA_UPSERT
    Kafka->>Neo4j: Apply idempotent graph mutations
    Kafka->>Spark: Deliver metadata event from checkpointed offset
    Spark->>Mongo: Replace/upsert document by file_id
    Parser->>SQLite: Commit new content_hash and graph IDs
```

## Ranh giới kiến trúc

| Layer | Trách nhiệm | Quy tắc phụ thuộc |
|---|---|---|
| `domain/` | Model, enum, event contract và lỗi nghiệp vụ. | Không phụ thuộc layer khác. |
| `parsing/` | AST, CFG, DFG, Call graph, Stable ID và diff. | Chỉ phụ thuộc `domain`. |
| `application/` | Use case service và port interface. | Giao tiếp qua ports, không khởi tạo adapter cụ thể. |
| `infrastructure/` | Kafka, JSONL writer, SQLite, config và filesystem adapters. | Implement ports từ application. |
| `cli/` | Composition root, load config và inject adapters. | Được phép nối các layer khi chạy command. |
| `spark_jobs/` | Spark Structured Streaming job cho MongoDB. | Độc lập với parser core. |

## Reflection

Sơ đồ kiến trúc cho thấy Lab 04 không chỉ là một parser cục bộ mà là pipeline streaming nhiều hệ thống. Điểm quan trọng nhất là tách graph events và metadata events thành hai nhánh ingestion khác nhau: Neo4j nhận topology trực tiếp qua Kafka Connect, còn MongoDB nhận metadata qua Spark để tận dụng checkpoint. Khi Task 4 đã hoàn tất, nhánh Neo4j có thể được xem là consumer idempotent của graph events, phù hợp với yêu cầu replay không sinh duplicate.